# Climate Data Extractor — J2000 → TALSIM-NG

**Purpose:** Converts the hourly climate output from the J2000 hydrological model
into the timestamp format and column selection required by TALSIM-NG's
time-series manager.

**What it does:**
- Parses `TimeLoop.dat` (J2000 output), skipping metadata headers
- Splits the combined ID column into separate `dates` and `time` columns
- Reformats timestamps from `YYYY-MM-DD` to `DD.MM.YYYY HH:MM`
- Lets you interactively choose which variables to export
  (e.g., `precip`, `tmean`, `potET`)
- Saves the full dataset and the filtered subset to Excel

**Input:** `TimeLoop.dat`  
**Output:** `processed climate data_all field.xlsx`,
            `processed climate data_required field for TALSIM.xlsx`

---

## Extracting climate data from .dat file generated by J2000

In [ ]:
import os
import pandas as pd
from io import StringIO
from collections import Counter

### File location in local

In [ ]:
file_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\climate_ziegenrueck\TimeLoop.dat"

In [ ]:
output_folder = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR"
output_filename_1 = "processed climate data_all field.xlsx"
output_filename_2 = "processed climate data_required field for TALSIM.xlsx"

os.makedirs(output_folder, exist_ok=True)
output_path_1 = os.path.join(output_folder, output_filename_1)
output_path_2 = os.path.join(output_folder, output_filename_2)

### Read the file

In [ ]:
with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()

start_index = next(i for i, line in enumerate(lines) if line.strip() == '@start') + 1
attributes_index = next(i for i, line in enumerate(lines) if line.strip().startswith('@attributes')) + 1

header = lines[attributes_index].strip().split('\t')
column_names = ['dates', 'time'] + header[1:]

print("Column names:")
print(header)

### Reading and Exporting data into csv or excel in gneral <br>
In the data frame of the text file (.dat) each attribute is identified as unique id which is the datetime stamp  and the header of that attribute is named as [ID] Each conatins several data including rainfall, evaporation, temperature etc. at hourly interval. We will divide each ID into two seperate column and name them as dates and time respectively. This will give us a range of data in one hour interval. 

In [ ]:
# Reading data using whitespace separator
data_str = ''.join(lines[start_index:])
data_io = StringIO(data_str)

df = pd.read_csv(
    data_io,
    sep=r'\s+',
    engine='python',
    names=column_names
)
print(df.head())


# Can be directly exported without any changes or modifications
#df.to_csv("climate_data_clean.csv")
#df.to_excel("climate_data_clean.xlsx")

### Extracting and modifying data set for TALSIM <br>
The data generated by  J2000 is usually shows the date as JJ:MM:TT format while the time as HH:MM in the column namely dates and time. However, the TALSIM-NG time series manager require the date and time in one single column as well as in a defined format [TT:MM:TT HH:MM]. Therefore the dataframe first need to change the date format and then put the dat time in seperate column.

In [ ]:
df = df[~df['dates'].astype(str).str.startswith('@')].copy()

def make_unique(names):
    counts = Counter()
    unique = []
    for name in names:
        counts[name] += 1
        suffix = f"_{counts[name] - 1}" if counts[name] > 1 else ""
        unique.append(f"{name}{suffix}")
    return unique

df.columns = make_unique(df.columns.tolist())

#combined datetime column
df['date_time_column'] = pd.to_datetime(df['dates'] + ' ' + df['time']).dt.strftime('%d.%m.%Y %H:%M')

#reordering columns
cols = df.columns.tolist()
reordered = cols[:2] + ['date_time_column'] + [c for c in cols[2:] if c not in ['date_time_column']]
df = df[reordered]


print(df.head())
df.to_csv("Processed climate data.csv", index=False)
#df.to_excel("Processed climate data.xlsx", index=False)'

#To store thefile in a define folder path
#df.to_excel(output_path_1, index=False)

#### Filtering only those column required for TALSIM-NG time series manager

In [ ]:
#pET mm/h

In [ ]:
print("\ncolumns in the dataset after processing :")
print(df.columns.tolist())

user_input = input("\nEnter the column names you want to extract (comma-separated): ")
selected_columns = [col.strip() for col in user_input.split(',')]
if 'date_time_column' not in selected_columns:
    selected_columns.insert(0, 'date_time_column')

# Check if all selected columns exist
missing = [col for col in selected_columns if col not in df.columns]

if missing:
    print(f"\n❌ These columns were not found in the dataset: {missing}")
else:
    # Filter the DataFrame
    filtered_df = df[selected_columns].copy()

    # Export filtered data
    output_file = "selected_columns_output.csv"
    filtered_df.to_csv(output_file, index=False)

    print("\nPreview of extracted data:")
    print(filtered_df.head()) 
#Export in excel  
filtered_df.to_excel(output_path_2, index=False)